# Time to accuracy
Wall-clock time of the decomposition-free $\mathrm{Sgn}$ (generalized Newton–Schulz with the $\mathrm{bpoly}(D)$ profile, power-iteration scaling) until the SVD-free residual $\|I - X_k^\top X_k\|_F/\sqrt{n}$ reaches the target $\varepsilon$, as a function of the degree $D$. For full-rank $M$ this residual bounds the relative Frobenius error against the exact $\mathrm{Sgn}(M)$ from above. No SVD is computed. The timed region holds the scaling and the iteration with its residual check (one host synchronization per step). Each of the `n_reps` repetitions uses a new random matrix of the same size and $\sigma_{\min}$, so the reported std is over matrices. Edit the config and rerun; set `devices=["cpu"]` without a GPU, and keep $\varepsilon$ above the rounding level of the dtype (about $10^{-6}$ in float32).

In [ ]:
import sys
sys.path.insert(0, "..")  # repo root, so ns_core and experiments import

import matplotlib.pyplot as plt
import torch

from ns_core import plots
from experiments.time_to_accuracy import (
    TimeToAccuracyConfig, run_time_to_accuracy, time_to_accuracy_table, iterations_table,
)
%matplotlib inline

In [ ]:
cfg = TimeToAccuracyConfig(
    sizes=[128, 256, 512], smins=[1e-2, 1e-4], degrees=[1, 2, 3, 4],
    devices=["cpu", "cuda"], dtypes=[torch.float32, torch.float64], eps=[1e-3, 1e-5],
    k_max=50, power_iters=10, power_margin=1.1, n_warmup=3, n_reps=20, cpu_threads=None, seed=0,
)
res = run_time_to_accuracy(cfg)

In [ ]:
# One selection: time to reach eps against D, for a chosen device, precision, size and conditioning
sel = dict(device="cpu", dtype=torch.float64, n=512, smin=1e-2, eps=1e-3)
ax = plots.plot_time_vs_degree(res, **sel)
ax.figure.tight_layout()

In [ ]:
# Precisions (or devices, sizes, ...) compared on one axis
fig, ax = plt.subplots(figsize=(5.5, 4))
for dtype in cfg.dtypes:
    plots.plot_time_vs_degree(res, "cpu", dtype, 512, 1e-2, 1e-3, ax=ax, label=str(dtype).removeprefix("torch."))
ax.set_title("cpu, $n$ = 512, $\\sigma_{\\min}$ = 0.01, $\\varepsilon$ = 0.001")
fig.tight_layout()

In [ ]:
# Tables for the same selection: time to reach eps, and iterations run
time_to_accuracy_table(res, sel["device"], sel["dtype"], sel["smin"], sel["eps"])
print()
iterations_table(res, sel["device"], sel["dtype"], sel["smin"], sel["eps"])